# 🍄 Gljive Hrvatske – vlastiti model za prepoznavanje

Ova bilježnica sama skine otvorene fotografije gljiva s **iNaturalista** (samo licence **CC0** i **CC BY**, koje dopuštaju i komercijalnu upotrebu), istrenira model za vrste iz atlasa i spremi ga u oblik koji aplikacija koristi.

**Kako pokrenuti**
1. Izbornik **Runtime → Change runtime type → T4 GPU → Save**.
2. Izbornik **Runtime → Run all**.
3. Kad pita, dopusti pristup **Google Driveu** (tamo se sve sprema, u mapu `GljiveModel`).
4. Pričekaj. Prvi put traje otprilike **3–5 sati** (skidanje popisa ~1 h, fotografija ~30 min, treniranje ~2–3 h).
5. Ako se Colab prekine ili zatvori: ponovno **Runtime → Run all**. Nastavlja gdje je stao.
6. Na kraju se u mapi `GljiveModel` na Driveu pojavi **`GljiveModel_v1.zip`**. Pošalji ga Claudeu i on ga ubaci u aplikaciju i usporedi s danskim modelom.

Licenca osnovnog modela: `efficientnet_b0.ra_in1k` iz biblioteke timm (Apache 2.0). Popis autora fotografija sprema se u `zasluge.csv`.

In [ ]:
# ===== POSTAVKE (ne treba mijenjati) =====
MODEL_NAME   = "efficientnet_b0.ra_in1k"  # osnovni model, iste veličine kao danski
IMG          = 224
MAX_PO_VRSTI = 300        # najviše fotografija po vrsti
MIN_PO_VRSTI = 30         # vrste s manje fotografija se preskaču
FOTO_PO_NALAZU = 2        # najviše fotografija s istog nalaza
LICENCE      = ["cc0", "cc-by"]   # samo licence koje dopuštaju i komercijalnu upotrebu
EPOHE        = 12
BATCH        = 128
LR           = 1e-3
VERZIJA      = 1
PRETRAINED   = True
API_PAUZA    = 1.0        # sekunde između upita iNaturalistu (njihovo pravilo: najviše ~1 u sekundi)

In [ ]:
%pip install -q timm onnx onnxruntime

In [ ]:
import os, io, json, time, csv, math, random, hashlib, tarfile, zipfile, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from PIL import Image
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT  = Path('/content/drive/MyDrive/GljiveModel')
    WORK = Path('/content/work')
except ImportError:  # Kaggle ili vlastito računalo
    OUT  = Path(os.environ.get('GH_OUT', '/kaggle/working/GljiveModel' if Path('/kaggle').exists() else 'GljiveModel'))
    WORK = Path(os.environ.get('GH_WORK', str(OUT.parent / 'gh_work')))
OUT.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
(OUT / 'popis').mkdir(exist_ok=True)
SLIKE = WORK / 'slike'; SLIKE.mkdir(exist_ok=True)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Spremam u:', OUT, '| uređaj:', DEV)
if DEV == 'cpu':
    print('⚠️ Nema GPU-a! Runtime → Change runtime type → T4 GPU, pa ponovno Run all. Bez GPU-a treniranje traje danima.')

In [ ]:
VRSTE = ["Boletus edulis", "Boletus reticulatus", "Boletus aereus", "Tylopilus felleus", "Rubroboletus satanas", "Cantharellus cibarius", "Hygrophoropsis aurantiaca", "Omphalotus olearius", "Macrolepiota procera", "Lepiota spp.", "Amanita phalloides", "Amanita virosa", "Amanita muscaria", "Amanita pantherina", "Amanita caesarea", "Amanita rubescens", "Morchella esculenta", "Gyromitra esculenta", "Pleurotus ostreatus", "Armillaria mellea", "Hypholoma fasciculare", "Galerina marginata", "Agaricus campestris", "Agaricus xanthodermus", "Russula virescens", "Russula emetica", "Lactarius deliciosus", "Craterellus cornucopioides", "Calocybe gambosa", "Inosperma erubescens", "Entoloma sinuatum", "Coprinus comatus", "Cortinarius orellanus", "Tuber magnatum", "Boletus pinophilus", "Butyriboletus regius", "Butyriboletus pseudoregius", "Butyriboletus appendiculatus", "Caloboletus calopus", "Neoboletus erythropus", "Suillellus luridus", "Suillellus queletii", "Imleria badia", "Xerocomus subtomentosus", "Xerocomellus chrysenteron", "Hortiboletus rubellus", "Gyroporus castaneus", "Gyroporus cyanescens", "Leccinum scabrum", "Leccinum pseudoscabrum", "Leccinum aurantiacum", "Leccinellum crocipodium", "Suillus luteus", "Suillus granulatus", "Suillus bovinus", "Suillus grevillei", "Suillus variegatus", "Chalciporus piperatus", "Strobilomyces strobilaceus", "Gomphidius glutinosus", "Craterellus tubaeformis", "Craterellus lutescens", "Hydnum repandum", "Sarcodon imbricatus", "Chlorophyllum rhacodes", "Macrolepiota mastoidea", "Lepiota cristata", "Lepiota clypeolaria", "Echinoderma asperum", "Agaricus augustus", "Agaricus bitorquis", "Agaricus sylvaticus", "Amanita verna", "Amanita citrina", "Amanita gemmata", "Amanita vaginata", "Coprinopsis atramentaria", "Coprinopsis picacea", "Clitocybe nebularis", "Infundibulicybe geotropa", "Infundibulicybe gibba", "Clitocybe odora", "Clitocybe rivulosa", "Clitopilus prunulus", "Lepista nuda", "Lepista glaucocana", "Lepista personata", "Marasmius oreades", "Tricholoma portentosum", "Tricholoma columbetta", "Tricholoma pardinum", "Tricholoma saponaceum", "Tricholoma sejunctum", "Tricholoma sciodes", "Tricholomopsis rutilans", "Melanoleuca melaleuca", "Lyophyllum decastes", "Lyophyllum fumosum", "Lyophyllum loricatum", "Leucocybe connata", "Laetiporus sulphureus", "Fistulina hepatica", "Grifola frondosa", "Meripilus giganteus", "Polyporus umbellatus", "Albatrellus cristatus", "Albatrellus pes-caprae", "Hapalopilus rutilans", "Auricularia auricula-judae", "Flammulina velutipes", "Hypholoma lateritium", "Psathyrella piluliformis", "Cyclocybe cylindracea", "Agrocybe praecox", "Desarmillaria tabescens", "Ganoderma lucidum", "Fomitopsis betulina", "Trametes versicolor", "Lentinus tigrinus", "Panus conchatus", "Oudemansiella mucida", "Hymenopellis radicata", "Gymnopus fusipes", "Rhodocollybia butyracea", "Megacollybia platyphylla", "Laccaria laccata", "Mycena pura", "Mycena inclinata", "Pluteus cervinus", "Volvariella bombycina", "Volvopluteus gloiocephalus", "Lactarius sanguifluus", "Lactarius deterrimus", "Lactarius semisanguifluus", "Lactarius chrysorrheus", "Lactarius torminosus", "Lactarius blennius", "Lactifluus piperatus", "Lactifluus vellereus", "Lactifluus volemus", "Russula cyanoxantha", "Russula grisea", "Russula vesca", "Russula heterophila", "Russula aurea", "Russula romellii", "Russula delica", "Russula nigricans", "Russula lepida", "Russula olivacea", "Cortinarius caperatus", "Cortinarius praestans", "Cortinarius bolaris", "Cortinarius semisanguineus", "Cortinarius trivialis", "Cortinarius rubellus", "Leucocortinarius bulbiger", "Pseudosperma rimosum", "Hebeloma sinapizans", "Paxillus involutus", "Tapinella atrotomentosa", "Entoloma clypeatum", "Entoloma sepium", "Hygrophorus marzuolus", "Hygrophorus russula", "Hygrophorus nemoreus", "Hygrophorus poetarum", "Cuphophyllus pratensis", "Hygrocybe punicea", "Stropharia aeruginosa", "Stropharia rugosoannulata", "Ramaria flava", "Ramaria botrytis", "Ramaria formosa", "Lycoperdon perlatum", "Apioperdon pyriforme", "Calvatia gigantea", "Scleroderma citrinum", "Phallus impudicus", "Clathrus ruber", "Morchella elata", "Pleurotus eryngii", "Tuber aestivum", "Fomitopsis pinicola", "Fomes fomentarius", "Coprinellus micaceus", "Stereum hirsutum", "Mycena galericulata", "Cuphophyllus virgineus", "Hygrocybe miniata", "Ganoderma applanatum", "Gymnopilus penetrans", "Armillaria lutea", "Phaeolepiota aurea", "Trametes hirsuta", "Trametes gibbosa", "Pseudocraterellus undulatus", "Schizophyllum commune", "Kuehneromyces mutabilis", "Laccaria amethystina", "Hygrocybe ceracea", "Xerocomellus pruinatus", "Hygrocybe conica", "Pholiota squarrosa", "Gliophorus psittacinus", "Phaeolus schweinitzii", "Tricholoma scalpturatum", "Russula ochroleuca", "Paralepista flaccida", "Ramaria stricta", "Tricholoma terreum", "Russula adusta", "Mycetinis alliaceus", "Lacrymaria lacrymabunda", "Ganoderma pfeifferi", "Hygrocybe coccinea", "Amanita fulva", "Chlorophyllum olivieri", "Coprinellus disseminatus", "Inocybe geophylla", "Mycena rosea", "Pseudoclitocybe cyathiformis", "Armillaria ostoyae", "Boletus ferrugineus", "Stropharia cyanea", "Amanita excelsa", "Mycena crocata", "Mycena haematopus", "Clavulina rugosa", "Hygrocybe acutoconica", "Clavulina coralloides", "Geastrum michelianum", "Clitocybe fragrans", "Laccaria proxima", "Mycena epipterygia", "Trametes ochracea", "Polyporus tuberaster", "Lactarius quietus", "Hygrocybe chlorophana", "Gymnopus dryophilus", "Caloboletus radicans", "Gymnopus confluens", "Agaricus impudicus", "Protostropharia semiglobata", "Lycoperdon nigrescens", "Tricholoma sulphureum", "Coprinopsis lagopus", "Panaeolus papilionaceus", "Thelephora terrestris", "Pseudohydnum gelatinosum", "Cortinarius elatior", "Marasmius rotula", "Hypholoma capnoides", "Pleurotus pulmonarius", "Pholiota gummosa", "Panaeolina foenisecii", "Hericium coralloides", "Rhodocollybia maculata", "Gliophorus laetus", "Hygrocybe cantharellus", "Lactarius necator", "Lycoperdon excipuliforme", "Strobilurus esculentus", "Mycena galopus", "Leucoagaricus nympharum", "Mycena vitilis", "Hortiboletus bubalinus", "Lactarius subdulcis", "Mycena tintinnabulum", "Tricholoma argyraceum", "Pluteus salicinus", "Lactarius pubescens", "Lactarius rufus", "Cortinarius largus", "Cortinarius malicorius", "Cortinarius torvus", "Cortinarius anserinus", "Cortinarius flexipes", "Mycena leptocephala", "Cortinarius croceus", "Agaricus sylvicola", "Crucibulum crucibuliforme", "Lepista sordida", "Pholiota adiposa", "Xerocomellus cisalpinus", "Gymnopus peronatus", "Hypholoma marginatum", "Sparassis crispa", "Russula paludosa", "Suillus cavipes", "Mycena pelianthina", "Mycena polygramma", "Mutinus caninus", "Lactarius helvus", "Lycoperdon utriforme", "Russula xerampelina", "Lactarius tabidus", "Russula depallens", "Ganoderma adspersum", "Lycoperdon pratense", "Cortinarius anomalus", "Hygrophorus pustulatus", "Scleroderma verrucosum", "Hygrocybe reidii", "Entoloma serrulatum", "Russula rosea", "Mycena sanguinolenta", "Leucoagaricus leucothites", "Hygrocybe insipida", "Mycena pseudocorticola", "Leccinum versipelle", "Gliophorus irrigatus", "Clitocybe metachroa", "Cortinarius alboviolaceus", "Tricholoma fulvum", "Russula praetervisa", "Xerocomellus porosporus", "Lentinellus ursinus", "Lactarius pyrogalus", "Russula atropurpurea", "Pleurotus dryinus", "Russula solaris", "Tricholoma ustale", "Auriscalpium vulgare", "Russula queletii", "Hygrophorus eburneus", "Psilocybe coronilla", "Gymnopus perforans", "Lactarius pallidus", "Agrocybe pediades", "Amanita porphyria", "Leucopaxillus giganteus", "Leccinum variicolor", "Mycena metata", "Cuphophyllus flavipes", "Gymnopus aquosus", "Russula claroflava", "Russula violeipes", "Cyathus striatus", "Mycena arcangeliana", "Pluteus phlebophorus", "Mycena aetites", "Melanoleuca cognata", "Mycena olivaceomarginata", "Hygrocybe nitrata", "Ampulloclitocybe clavipes", "Lepiota magnispora", "Russula chloroides", "Russula fragilis", "Paxillus obscurisporus", "Pluteus romellii", "Scleroderma areolatum", "Coltricia perennis", "Cyanoboletus pulverulentus", "Hygrophorus olivaceoalbus", "Chrysomphalina grossula", "Clitopilus hobsonii", "Lactarius fluens", "Lepiota castanea", "Panaeolus acuminatus", "Russula velenovskyi", "Macrolepiota excoriata", "Hebeloma mesophaeum", "Pholiota lenta", "Paxillus filamentosus", "Lentinellus cochleatus", "Russula risigallina", "Cortinarius infractus", "Clitocybe phyllophila", "Coprinopsis nivea", "Mycena tenerrima", "Panaeolus semiovatus", "Russula foetens", "Lactarius aurantiacus", "Russula grata", "Hemipholiota populnea", "Leccinum cyaneobasileucum", "Russula nobilis", "Lactarius camphoratus", "Lactarius romagnesii", "Russula densifolia", "Russula cessans", "Laccaria bicolor", "Lycoperdon echinatum", "Russula aurora", "Inocybe sindonia", "Hericium cirrhatum", "Mycena abramsii", "Lactarius circellatus", "Conocybe aporos", "Flammulina elastica", "Lactarius vietus", "Omphalina pyxidata", "Mycena acicula", "Clavariadelphus pistillaris", "Cortinarius obtusus", "Amanita aureola", "Amanita eliae", "Amanita beckeri", "Amanita spissa", "Amanita ovoidea", "Amanita echinocephala", "Amanita aspera", "Amanita inaurata", "Amanita livido", "Amanita umbrinolutea", "Amanita crocea", "Volvariella surrecta", "Lepiota ventriosospora", "Lepiota alba", "Lepiota naucina", "Leucoagaricus holosericeus", "Leucocoprinus badhamii", "Cystoderma amianthinum", "Chamaemyces fracidus", "Limacella illinita", "Limacella guttata", "Catathelasma imperiale", "Coprinus truncorum", "Coprinus hemerobius", "Coprinus cinereus", "Agaricus macrosporus", "Agaricus bisporus", "Agaricus arvensis", "Agaricus abruptibulbus", "Agaricus perrarus", "Agaricus benesii", "Agaricus langei", "Agaricus vaporarius", "Agaricus radicatus", "Agaricus purpurascens", "Panaeolus sphinctrinus", "Panaeolus rickenii", "Anellaria semiovata", "Psathyrella candolleana", "Psathyrella prona", "Psathyrella spadiceo-grisea", "Psathyrella velutina", "Stropharia squamosa", "Stropharia eximia", "Agrocybe dura", "Agrocybe erebia", "Agrocybe paludosa", "Pholiota squarroso-adiposa", "Pholiota aurivella", "Pholiota destruens", "Pholiota flammans", "Pholiota carbonaria", "Pholiota flavida", "Pholiota amara", "Pholiotina blattaria", "Bolbitius vitellinus", "Gymnopilus spectabilis", "Hebeloma radicosum", "Hebeloma crustuliniforme", "Cortinarius traganus", "Cortinarius coerulescens", "Cortinarius nebularis", "Cortinarius zinziberatus", "Cortinarius collinitus", "Cortinarius violaceus", "Cortinarius arcuatorum", "Cortinarius glaucopus", "Cortinarius bulliardi", "Cortinarius hinnuleus", "Cortinarius pholideus", "Cortinarius nemorensis", "Dermocybe cinnamomea", "Dermocybe anthracina", "Dermocybe cinnabarina", "Inocybe maculata", "Inocybe grammata", "Inocybe umbratica", "Inocybe sambucina", "Inocybe godeyi", "Inocybe cincinnata", "Inocybe hirsuta", "Entoloma aprile", "Entoloma nitidum", "Entoloma mougeoti", "Entoloma mammosum", "Entoloma staurosporum", "Clitopilus cretatus", "Rhodocybe truncata", "Pluteus atricapillus", "Pluteus leoninus", "Collybia dryophila", "Collybia platyphylla", "Collybia exculpta", "Collybia acervata", "Collybia peronata", "Collybia hariolorum", "Oudemansiella longipes", "Strobilurus stephanocystis", "Strobilurus tenacellus", "Marasmius scorodonius", "Marasmius wynnei", "Marasmius bulliardii", "Marasmius androsaceus", "Marasmiellus ramealis", "Micromphale brassicolens", "Micromphale foetidum", "Mycena rosella", "Mycena callopoda", "Mycena flavipes", "Leucopaxillus amarus", "Tricholoma georgi", "Tricholoma viridilutescens", "Tricholoma flavovirens", "Tricholoma auratum", "Tricholoma albobrunneum", "Tricholoma batschii", "Tricholoma bufonium", "Tricholoma imbricatum", "Tricholoma vaccinum", "Tricholoma virgatum", "Tricholoma tigrinum", "Tricholoma squarrulosum", "Tricholoma atrosquamosum", "Tricholoma orirubens", "Tricholoma album", "Tricholoma acerbum", "Lepista irina", "Lepista caespitosa", "Lepista rickenii", "Melanoleuca grammopodia", "Melanoleuca subbrevipes", "Melanoleuca strictipes", "Melanoleuca excissa", "Clitocybe inversa", "Clitocybe gilva", "Clitocybe vermicularis", "Clitocybe maxima", "Clitocybe costata", "Clitocybe alexandri", "Clitocybe inornata", "Clitocybe cerussata", "Lyophyllum immundum", "Lyophyllum infumatum", "Lyophyllum semitale", "Lactarius porninsis", "Lactarius cilicioides", "Lactarius scrobiculatus", "Lactarius zonarius", "Lactarius luteolus", "Lactarius trivialis", "Lactarius contorversus", "Lactarius pergamenus", "Lactarius fuliginosus", "Lactarius picinus", "Lactarius uvidus", "Russula aeruginea", "Russula parazurea", "Russula amoena", "Russula integra", "Russula mustelina", "Russula vitellina", "Russula fellea", "Russula illota", "Russula laurocerasi", "Russula pseudodelica", "Russula viscida", "Russula amethystina", "Russula puellaris", "Russula versicolor", "Russula acrifolia", "Russula pectinatoides", "Russula amoenolens", "Russula sanguinea", "Russula badia", "Asterophora lycoperdoides", "Hygrophorus pudorinus", "Hygrophorus penarius", "Hygrophorus leporinus", "Hygrophorus queletii", "Hygrophorus chrysodon", "Hygrophorus co", "Hygrophorus gliocyclus", "Hygrophorus dichrous", "Hygrophorus agathosmus", "Hygrophorus lucorum", "Hygrophorus hypotheius", "Hygrophorus leucophaeus", "Hygrophorus erubescens", "Hygrophorus discoideus", "Hygrophorus aureus", "Hygrocybe konradii", "Hygrocybe obrussea", "Hygrocybe brevispora", "Hygrocybe citrina", "Hygrocybe quieta", "Hygrocybe intermedius", "Hygrocybe coccineus", "Hygrocybe nigrescens", "Hygrocybe laeta", "Hygrocybe reai", "Hygrocybe ovina", "Camarophyllus niveus", "Camarophyllus russocoriaceus", "Cantharellus friesii", "Cantharellus amethysteus", "Cantharellus subcibarius", "Gomphus clavatus", "Cantharellus cinerus", "Xeromphalina campanella", "Gomphidius maculatus", "Chroogomphus rutilus", "Chroogomphus helveticus", "Pleurotus columbinus", "Pleurotus sapidus", "Pleurocybella porrigens", "Panus tigrinus", "Crepidotus variabilis", "Phylloporus rhodoxanthus", "Boletus fechtneri", "Boletus impolitus", "Boletus fragrans", "Boletus junquilleus", "Boletus albidus", "Boletus satanoides", "Boletus rhodoxanthus", "Boletus purpureus", "Boletus dupainii", "Xerocomus spadiceus", "Xerocomus parasticus", "Leccinum testaceo-scabrum", "Leccinum criseum", "Leccinum holopus", "Leccinum duriusculum", "Suillus aeruginascens", "Suillus placidus", "Gyrodon lividus", "Polyporus confluens", "Polyporus ovinus", "Polyporus cristatus", "Polyporus pes-caprae", "Polyporus giganteus", "Polyporus montanus", "Polyporus frandosus", "Polyporus intybaceus", "Polyporus squamosus", "Polyporus lentus", "Polyporus sulphureus", "Polyporus lepideus", "Polyporus mori", "Polyporus arcularius", "Polyporus badius", "Merulius tremellosus", "Inonotus dryadeus", "Osmoporus odoratus", "Daedalea quercina", "Daedalea confragosa", "Thelephora palmata", "Sarcodon badium", "Hydnum rufescens", "Calodon aurantiacum", "Calodon compactum", "Dryodon erinaceus", "Pleurodon auriscalpium", "Sparassis laminosa", "Clavaria flava", "Clavaria aurea", "Clavaria rufescens", "Clavaria formosa", "Clavaria pallida", "Clavaria lavendula", "Clavaria cristata", "Clavaria rugosa", "Clavaria fennica", "Clavaria invalii", "Clavaria stricta", "Clavaria vermicularis", "Pterula multifida", "Calocera viscosa", "Clavaria pistillaris", "Clavaria truncata", "Tremellodon gelatinosum", "Tremella mesenterica", "Exidia galndulosa", "Tremiscus helvelloides", "Lycoperdon caelatum", "Lycoperdon mammaeforme", "Lycoperdon saccatum", "Lycoperdon umbrinum", "Bovista plumbea", "Rhizopogon roseolus", "Scleroderma geaster", "Geastrum triplex", "Geastrum fimbriatum", "Geastrum pectinatum", "Astraeus hygrometricus", "Clathrus cancellatus", "Phallus hadriani", "Sebacina laciniata", "Spathularia flavida", "Bulgaria inquinans", "Lycogala epidendron", "Arcyria denudata", "Tubifera ferruginosa", "Fuligo septica", "Xylaria polymorpha", "Xylaria hypoxylon", "Morchella deliciosa", "Morchella pragensis", "Mitrophora hybrida", "Verpa bohemica", "Verpa digitaliformis", "Helvella crispa", "Helvella lacunosa", "Helvella elastica", "Gyromitra fastigiata", "Tuber brumale", "Tuber melanosporum", "Elaphomyces granulatus", "Choiromyces maeandriformis", "Otidea onotica", "Caloscypha fulgens", "Aleuria aurantia", "Sarcoscypha coccinea", "Scutellinia scutellata", "Paxina acetabulum", "Peziza vesiculosa", "Peziza badia", "Discina perlata", "Disciotis venosa", "Sclerotinia tuberosa", "Agaricus fuscofibrillosus", "Aleuria boudieri", "Aleuria flavorubens", "Aleurocystidiellum subcruentatum", "Amanita friabilis", "Amanita lepiotoides", "Amanita pachyvolvata", "Amanita solitaria", "Amanita vittadinii", "Amylocorticium subincarnatum", "Amylocystis lapponicus", "Anomoporia bombycina", "Antrodia crassa", "Antrodia variiformis", "Antrodiella citrinella", "Antrodiella fragrans", "Arrhenia spathulata", "Ascobolus behnitziensis", "Ascobolus epimyces", "Ascotremella faginea", "Atheloderma mirabile", "Auriporia aurulenta", "Baeospora myriadophylla", "Bisporella calycellinoides", "Bolbitius lacteus", "Boletus adonis", "Boletus depilatus", "Boletus torosus", "Callistosporium donadinii", "Callistosporium elaeodes", "Callistosporium luteoolivaceum", "Calocybe carnea", "Calocybe constricta", "Calocybe ionides", "Calocybe obscurissima", "Camarophyllopsis foetens", "Camarophyllopsis micacea", "Camarophyllopsis phaeophylla", "Campanella caesia", "Ceriporiopsis jelicii", "Ceriporiopsis myceliosa", "Ceriporiopsis subrufa", "Chalciporus rubinus", "Cheilymenia magnifica", "Ciboria aestivalis", "Clavulicium macounii", "Clitocybe collina", "Clitocybula lacerata", "Contumyces rosellus", "Coprinus strossmayeri", "Cordyceps bifusispora", "Cordyceps intermedia", "Cordyceps longisegmentis", "Cordyceps riverae", "Cordyceps sphecocephala", "Cortinarius bulliardii", "Cortinarius ionochlorus", "Cotylidia pannosa", "Crinipellis subtomentosa", "Cudonia circinans", "Cystoderma superbum", "Dacryobolus karstenii", "Dentipellis fragilis", "Dermoloma cuneifolium", "Dermoloma josserandii", "Dermoloma pseudocuneifolium", "Desmazierella acicola", "Discina fastigiata", "Discina leucoxantha", "Discina montana", "Discina parma", "Disciseda bovista", "Disciseda candida", "Elaphomyces anthracinus", "Endoptychum agaricoides", "Entoloma bloxamii", "Entoloma caesiocinctum", "Entoloma catalaunicum", "Entoloma corvinum", "Entoloma cyanulum", "Entoloma kervernii", "Entoloma opacum", "Entoloma plebeioides", "Entoloma plebejum", "Entoloma porphyrophaeum", "Entoloma pseudoturci", "Entoloma reinwaldii", "Entoloma rhombisporum", "Entoloma saundersii", "Faerberia carbonaria", "Flammulina ononidis", "Flavophlebia sulphureoisabellina", "Fomitopsis rosea", "Fomitopsis spraguei", "Galerina jaapii", "Galerina paludosa", "Galerina tibiicystis", "Gastrosporium simplex", "Geastrum minimum", "Geoglossum cookeanum", "Geoglossum glutinosum", "Geoglossum umbratile", "Geopora nicaeensis", "Geopyxis majalis", "Gerhardtia piperata", "Gloeocystidiellum subasperisporum", "Gomphidius roseus", "Gymnopilus bellulus", "Gymnopilus picreus", "Haasiella splendidissima", "Haasiella venustissima", "Hapalopilus croceus", "Hapalopilus salmonicolor", "Hebeloma gigaspermum", "Helvella albella", "Helvella branzeziana", "Helvella cupuliformis", "Helvella helvellula", "Helvella lactea", "Helvella phlebophora", "Helvella queletiana", "Helvella spadicea", "Hericium flagellum", "Hexagonia nitida", "Heyderia abietis", "Holwaya mucida", "Humaria aurantia", "Hydropus atramentosus", "Hygrocybe calciphila", "Hygrocybe calyptriformis", "Hygrocybe citrinovirens", "Hygrocybe coccineocrenata", "Hygrocybe colemanniana", "Hygrocybe fornicata", "Hygrocybe ingrata", "Hygrocybe intermedia", "Hygrocybe irrigata", "Hygrocybe lacmus", "Hygrocybe perplexa", "Hygrocybe russocoriacea", "Hygrocybe sciophanoides", "Hygrocybe spadicea", "Hygrocybe splendidissima", "Hygrocybe subpapillata", "Hygrophorus calophyllus", "Hygrophorus camarophyllus", "Hygrophorus capreolarius", "Hygrophorus hyacinthinus", "Hygrophorus hypothejus", "Hygrophorus mesotephrus", "Hyphoderma cremeoalbum", "Hyphoderma guttuliferum", "Hyphoderma litschaueri", "Hyphoderma macedonicum", "Hypholoma elongatum", "Hypholoma myosotis", "Inonotus dryophilus", "Inonotus nidus-pici", "Jaapia ochroleuca", "Junghuhnia collabens", "Junghuhnia fimbriatella", "Kuehneromyces lignicola", "Laccaria pumila", "Lactarius acris", "Lactarius cistophilus", "Lactarius lacunarum", "Lactarius lilacinus", "Lactarius omphaliformis", "Lactarius tesquorum", "Lambertella corni-maris", "Lentinus degener", "Leptosporomyces roseus", "Leucoagaricus pilatianus", "Leucogaster nudus", "Leucopaxillus compactus", "Leucopaxillus macrocephalus", "Lyophyllum rhopalopodium", "Lyophyllum transforme", "Marasmiellus humillimus", "Marasmius anomalus", "Marasmius buxi", "Marasmius caricis", "Marasmius collinus", "Marasmius cornelii", "Marasmius hudsonii", "Marasmius menieri", "Marasmius undatus", "Melanogaster vittadinii", "Metulodontia nivea", "Microglossum olivaceum", "Microglossum viride", "Mitrula paludosa", "Moellerodiscus iodotingens", "Moellerodiscus tenuistipes", "Mollisia olivascens", "Mucronella bresadolae", "Mucronella calva", "Mycena latifolia", "Mycenastrum corium", "Myriosclerotinia dennisii", "Myriostoma coliforme", "Oligoporus cerifluus", "Oligoporus lowei", "Oligoporus placenta", "Ombrophila rivulorum", "Omphalina baeospora", "Omphalina chrysophylla", "Omphalina obatra", "Onygena equina", "Orbilia polyspora", "Ossicaulis lignatilis", "Otidea propinquata", "Pachyella peltata", "Pachyella pseudosuccosa", "Pachyella punctispora", "Pachyella violaceonigra", "Parmastomyces mollissimus", "Peziza alborosea", "Peziza ammophila", "Peziza apiculata", "Peziza boltonii", "Peziza buxea", "Peziza flavida", "Peziza merdae", "Peziza muscicola", "Peziza nivalis", "Peziza obtusapiculata", "Peziza saccardiana", "Peziza subuliginosa", "Peziza subumbrina", "Phaeocollybia jennyi", "Phaeomarasmius limulatus", "Phaeomarasmius muricatus", "Phallogaster saccatus", "Phellinus nigrolimitatus", "Phellinus pouzarii", "Phlebia albida", "Phlebia centrifuga", "Phlebia cornea", "Phlebia griseoflavescens", "Phlebia subulata", "Phylloporus pelletieri", "Phyllotopsis nidulans", "Piptoporus quercinus", "Piptoporus soloniensis", "Pisolithus arhizus", "Plectania melastoma", "Plectania platensis", "Pleurotus calyptratus", "Pleurotus cornucopiae", "Poronia punctata", "Porphyrellus porphyrosporus", "Psathyrella ammophila", "Psathyrella melanthina", "Psathyrella typhae", "Pseudoomphalina kalchbrenneri", "Pseudoplectania melaena", "Pseudorhizina sphaerospora", "Psilopezia nummularialis", "Ptychoverpa bohemica", "Pulveroboletus hemichrysus", "Pulveroboletus lignicola", "Pulvinula globifera", "Pulvinula laeterubra", "Pulvinula ovalispora", "Pycnopeziza sejournei", "Pycnoporellus fulgens", "Pyronema dubium", "Resupinatus vetlinianus", "Rhodocybe fallax", "Rhodoscypha ovilla", "Rigidoporus crocatus", "Rodwayella sessilis", "Russula alnetorum", "Sarcoscypha macaronesica", "Scleroderma polyrhizum", "Scutellinia pilatii", "Scutellinia tuberculata", "Scytinostroma galactinum", "Scytinostroma odoratum", "Scytinostromella heterogenea", "Skeletocutis odora", "Skeletocutis papyracea", "Skeletocutis stellae", "Sparassis spathulata", "Spongipellis delectans", "Steccherinum subcrinale", "Stropharia dorsipora", "Strossmayeria rackii", "Suillus lakei", "Suillus tridentinus", "Tatraea dumbirensis", "Trichoglossum confusum", "Trichoglossum hirsutum", "Trichoglossum variabile", "Tricholoma aurantium", "Tricholoma caligatum", "Tricholoma colossus", "Tricholoma goniospermum", "Tricholoma luridum", "Tulostoma fimbriatum", "Tulostoma melanocyclum", "Urnula craterium", "Veluticeps ambigua", "Verpa conica", "Vibrissea truncorum", "Xerocomus moravicus", "Xerula caussei", "Psilocybe semilanceata", "Psilocybe serbica", "Psilocybe cyanescens", "Panaeolus cinctulus"]
print(len(VRSTE), 'vrsta u atlasu')

In [ ]:
# ----- 1. Pronađi svaku vrstu na iNaturalistu -----
S = requests.Session()
S.headers['User-Agent'] = 'GljiveHrvatske-trening/1.0 (github.com/markofrankovic-afk/Gljive-Hrvatske)'
API = 'https://api.inaturalist.org/v1/'

def api(path, **params):
    for pokusaj in range(6):
        try:
            r = S.get(API + path, params=params, timeout=60)
            time.sleep(API_PAUZA)
            if r.status_code == 200:
                return r.json()
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(30 * (pokusaj + 1)); continue
            print('  upit nije uspio', r.status_code, path, params); return None
        except requests.RequestException as e:
            print('  mreža:', e); time.sleep(20 * (pokusaj + 1))
    return None

TAKS_F = OUT / 'taksoni.json'
TAKS = json.loads(TAKS_F.read_text()) if TAKS_F.exists() else {}
RANKS = {'species', 'subspecies', 'variety', 'form', 'complex', 'hybrid'}
nove = [v for v in VRSTE if v not in TAKS]
for i, ime in enumerate(nove):
    j = api('taxa', q=ime, per_page=30) or {}
    hit = None
    for r in j.get('results', []):
        if r.get('rank') in RANKS and r.get('iconic_taxon_name') in (None, 'Fungi', 'Protozoa') and \
           (r.get('name', '').lower() == ime.lower() or (r.get('matched_term') or '').lower() == ime.lower()):
            hit = {'id': r['id'], 'name': r['name'], 'rank': r['rank'], 'obs': r.get('observations_count', 0)}
            break
    TAKS[ime] = hit
    if i % 25 == 0 or i == len(nove) - 1:
        TAKS_F.write_text(json.dumps(TAKS, ensure_ascii=False))
        print(f'{i+1}/{len(nove)}', ime, '->', hit and hit['name'])
nema = [v for v in VRSTE if not TAKS.get(v)]
print('Pronađeno', len(VRSTE) - len(nema), '/', len(VRSTE), '| nije pronađeno:', len(nema))
print(nema[:40])

In [ ]:
# ----- 2. Popis fotografija sa slobodnom licencom za svaku vrstu -----
def popis_za(ime, t):
    f = OUT / 'popis' / f"{t['id']}.json"
    if f.exists():
        return json.loads(f.read_text())
    fotke, page = [], 1
    while len(fotke) < MAX_PO_VRSTI * 1.15 and page <= 8:
        j = api('observations', taxon_id=t['id'], quality_grade='research', photos='true',
                photo_license=','.join(LICENCE), per_page=200, page=page, order_by='id', order='desc')
        if j is None:
            return None  # pokušaj opet pri sljedećem pokretanju
        res = j.get('results', [])
        for o in res:
            n = 0
            for p in o.get('photos', []):
                lic = (p.get('license_code') or '').lower()
                if lic not in LICENCE or not p.get('url'):
                    continue
                url = p['url'].replace('/square.', '/medium.')
                fotke.append({'pid': p['id'], 'oid': o['id'], 'ot': (o.get('taxon') or {}).get('id'),
                              'url': url, 'lic': lic, 'attr': p.get('attribution', ''),
                              'user': (o.get('user') or {}).get('login', '')})
                n += 1
                if n >= FOTO_PO_NALAZU: break
        if len(res) < 200: break
        page += 1
    f.write_text(json.dumps(fotke, ensure_ascii=False))
    return fotke

POPIS = {}
lista = [(v, TAKS[v]) for v in VRSTE if TAKS.get(v)]
for i, (ime, t) in enumerate(lista):
    p = popis_za(ime, t)
    if p is not None:
        POPIS[ime] = p
    if i % 25 == 0 or i == len(lista) - 1:
        print(f'{i+1}/{len(lista)}', ime, len(p or []))
print('Ukupno fotografija:', sum(len(v) for v in POPIS.values()))

In [ ]:
# ----- 3. Razredi: spoji sinonime, makni duplikate, preskoči vrste s premalo fotografija -----
po_taks = {}
for ime in VRSTE:
    t = TAKS.get(ime)
    if t and ime in POPIS:
        po_taks.setdefault(t['id'], []).append(ime)
vlasnik = {}   # pid -> taksonski id razreda (precizniji takson ima prednost)
for tid, imena in po_taks.items():
    for p in POPIS[imena[0]]:
        if p['pid'] not in vlasnik or p['ot'] == tid:
            vlasnik[p['pid']] = tid
RAZREDI, PRESKOCENE = [], []
for tid, imena in po_taks.items():
    fot = [p for p in POPIS[imena[0]] if vlasnik.get(p['pid']) == tid][:MAX_PO_VRSTI]
    if len(fot) >= MIN_PO_VRSTI:
        RAZREDI.append({'ime': imena[0], 'sinonimi': imena[1:], 'tid': tid, 'fotke': fot})
    else:
        PRESKOCENE.append((imena[0], len(fot)))
RAZREDI.sort(key=lambda r: r['ime'])
print('Razreda za učenje:', len(RAZREDI), '| preskočeno (premalo slobodnih fotografija):', len(PRESKOCENE))
print('Fotografija:', sum(len(r['fotke']) for r in RAZREDI))

In [ ]:
# ----- 4. Skidanje fotografija (spremaju se smanjene, kratka strana 288 px) -----
TAR = OUT / 'slike.tar'
if TAR.exists() and not any(SLIKE.iterdir()):
    print('Raspakiravam ranije skinute fotografije s Drivea…')
    with tarfile.open(TAR) as t:
        try: t.extractall(WORK, filter='data')
        except TypeError: t.extractall(WORK)

def skini(p):
    f = SLIKE / f"{p['pid']}.jpg"
    if f.exists():
        return True
    for pokusaj in range(3):
        try:
            r = requests.get(p['url'], timeout=30)
            if r.status_code != 200:
                time.sleep(2); continue
            im = Image.open(io.BytesIO(r.content)).convert('RGB')
            k = 288 / min(im.size)
            if k < 1:
                im = im.resize((max(1, round(im.width * k)), max(1, round(im.height * k))), Image.BICUBIC)
            im.save(f, 'JPEG', quality=88)
            return True
        except Exception:
            time.sleep(2)
    return False

sve = [p for r in RAZREDI for p in r['fotke']]
treba = [p for p in sve if not (SLIKE / f"{p['pid']}.jpg").exists()]
print('Treba skinuti', len(treba), 'od', len(sve))
ok = 0
with ThreadPoolExecutor(16) as ex:
    for i, res in enumerate(ex.map(skini, treba)):
        ok += res
        if i % 2000 == 0: print(f'  {i}/{len(treba)}')
print('Skinuto', ok, '| nije uspjelo', len(treba) - ok)
if treba and ok:
    print('Spremam kopiju fotografija na Drive (da sljedeći put ne treba skidati)…')
    with tarfile.open(TAR, 'w') as t: t.add(SLIKE, arcname='slike')

In [ ]:
# ----- 5. Podjela na učenje i provjeru (10 % nalaza ide u provjeru, po nalazu da nema prepisivanja) -----
def u_provjeri(oid):
    return int(hashlib.md5(str(oid).encode()).hexdigest(), 16) % 10 == 0
TRAIN, VAL = [], []
for ci, r in enumerate(RAZREDI):
    for p in r['fotke']:
        f = SLIKE / f"{p['pid']}.jpg"
        if f.exists():
            (VAL if u_provjeri(p['oid']) else TRAIN).append((str(f), ci))
LABELS = [r['ime'] for r in RAZREDI]
print('Učenje:', len(TRAIN), '| provjera:', len(VAL), '| razreda:', len(LABELS))

In [ ]:
# ----- 6. Treniranje -----
import timm
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

model = timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=len(LABELS))
cfg = model.pretrained_cfg
MEAN, STD = list(cfg.get('mean', (0.485, 0.456, 0.406))), list(cfg.get('std', (0.229, 0.224, 0.225)))
CROP = 0.875
tr_train = T.Compose([T.RandomResizedCrop(IMG, scale=(0.3, 1.0), interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomHorizontalFlip(), T.ColorJitter(0.25, 0.25, 0.15, 0.0),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
tr_val = T.Compose([T.Resize(round(IMG / CROP), interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(IMG),
                    T.ToTensor(), T.Normalize(MEAN, STD)])

class DS(Dataset):
    def __init__(s, items, tf): s.items, s.tf = items, tf
    def __len__(s): return len(s.items)
    def __getitem__(s, i):
        f, c = s.items[i]
        try: im = Image.open(f).convert('RGB')
        except Exception: im = Image.new('RGB', (IMG, IMG))
        return s.tf(im), c

NW = min(8, os.cpu_count() or 2)
dl_tr = DataLoader(DS(TRAIN, tr_train), batch_size=BATCH, shuffle=True, num_workers=NW, pin_memory=True, drop_last=True, persistent_workers=NW > 0)
dl_va = DataLoader(DS(VAL, tr_val), batch_size=BATCH, shuffle=False, num_workers=NW, pin_memory=True)

model = model.to(DEV).to(memory_format=torch.channels_last)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.02)
steps = EPOHE * len(dl_tr); warm = len(dl_tr)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
scaler = torch.amp.GradScaler('cuda', enabled=DEV == 'cuda')
crit = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

CKPT, BEST = OUT / 'ckpt.pt', OUT / 'najbolji.pt'
pocetak, najbolji = 0, 0.0
if CKPT.exists():
    ck = torch.load(CKPT, map_location=DEV, weights_only=False)
    if ck.get('labels') == LABELS:
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
        scaler.load_state_dict(ck['scaler']); pocetak, najbolji = ck['epoha'] + 1, ck['najbolji']
        print('Nastavljam od epohe', pocetak + 1)
    else:
        print('Popis vrsta se promijenio, treniram ispočetka.')

def provjera():
    model.eval(); t1 = t3 = n = 0
    with torch.no_grad(), torch.autocast(DEV, enabled=DEV == 'cuda'):
        for x, y in dl_va:
            x, y = x.to(DEV).to(memory_format=torch.channels_last), y.to(DEV)
            top = model(x).topk(3, 1).indices
            t1 += (top[:, 0] == y).sum().item(); t3 += (top == y[:, None]).any(1).sum().item(); n += len(y)
    model.train(); return t1 / max(1, n), t3 / max(1, n)

for ep in range(pocetak, EPOHE):
    model.train(); t0 = time.time(); zbroj = 0.0
    for i, (x, y) in enumerate(dl_tr):
        x, y = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last), y.to(DEV, non_blocking=True)
        with torch.autocast(DEV, enabled=DEV == 'cuda'):
            loss = crit(model(x), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        zbroj += loss.item()
        if i % 200 == 0: print(f'  epoha {ep+1} korak {i}/{len(dl_tr)} gubitak {loss.item():.3f}')
    a1, a3 = provjera()
    print(f'Epoha {ep+1}/{EPOHE}: gubitak {zbroj/max(1,len(dl_tr)):.3f} | točno {a1*100:.1f} % | među prva 3: {a3*100:.1f} % | {(time.time()-t0)/60:.1f} min')
    if a1 >= najbolji:
        najbolji = a1; torch.save({'model': model.state_dict(), 'labels': LABELS, 'top1': a1, 'top3': a3}, BEST)
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(), 'scaler': scaler.state_dict(),
                'epoha': ep, 'najbolji': najbolji, 'labels': LABELS}, CKPT)
print('Najbolja točnost na provjeri:', round(najbolji * 100, 1), '%')

In [ ]:
# ----- 7. Izvoz za aplikaciju (ONNX) i paket za Claudea -----
import onnx, onnxruntime as ort, numpy as np
best = torch.load(BEST, map_location='cpu', weights_only=False)
m = timm.create_model(MODEL_NAME, pretrained=False, num_classes=len(LABELS)); m.load_state_dict(best['model']); m.eval()
PAKET = OUT / 'paket'; shutil.rmtree(PAKET, ignore_errors=True); PAKET.mkdir()
onx = PAKET / 'model.onnx'; x = torch.randn(1, 3, IMG, IMG)
kw = dict(input_names=['input'], output_names=['logits'], opset_version=17, do_constant_folding=True)
try:
    torch.onnx.export(m, x, str(onx), dynamo=False, **kw)
except TypeError:
    torch.onnx.export(m, x, str(onx), **kw)
onnx.checker.check_model(str(onx))
razlika = np.abs(ort.InferenceSession(str(onx)).run(None, {'input': x.numpy()})[0] - m(x).detach().numpy()).max()
print('ONNX provjera, najveća razlika:', float(razlika))
assert razlika < 1e-2

data = onx.read_bytes(); onx.unlink(); DIO = 12_000_000; parts, sizes = [], []
for k in range(0, len(data), DIO):
    n = f'model.onnx.{k // DIO:03d}'; (PAKET / n).write_bytes(data[k:k + DIO]); parts.append(n); sizes.append(len(data[k:k + DIO]))
meta = {'model': f'gh-{MODEL_NAME}-v{VERZIJA}', 'input_size': [3, IMG, IMG], 'mean': MEAN, 'std': STD, 'crop_pct': CROP,
        'interpolation': 'bicubic', 'num_classes': len(LABELS), 'verzija': VERZIJA, 'vlastiti': True,
        'licencija': 'Vlastiti model Gljiva Hrvatske. Osnova: timm efficientnet_b0 (Apache 2.0). Fotografije: iNaturalist, CC0 i CC BY (autori u zasluge.csv).',
        'top1_provjera': round(best['top1'], 4), 'top3_provjera': round(best['top3'], 4), 'parts': parts, 'part_sizes': sizes}
(PAKET / 'meta.json').write_text(json.dumps(meta, ensure_ascii=False, indent=1))
(PAKET / 'labels.json').write_text(json.dumps(LABELS, ensure_ascii=False))
with open(PAKET / 'zasluge.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['vrsta', 'foto_id', 'nalaz', 'autor', 'licenca', 'navod'])
    for r in RAZREDI:
        for p in r['fotke']:
            if (SLIKE / f"{p['pid']}.jpg").exists():
                w.writerow([r['ime'], p['pid'], f"https://www.inaturalist.org/observations/{p['oid']}", p['user'], p['lic'], p['attr']])
(PAKET / 'izvjestaj.json').write_text(json.dumps({
    'razreda': len(LABELS), 'ucenje': len(TRAIN), 'provjera': len(VAL), 'top1': best['top1'], 'top3': best['top3'],
    'sinonimi': {r['ime']: r['sinonimi'] for r in RAZREDI if r['sinonimi']},
    'preskocene': PRESKOCENE, 'nije_pronadeno': [v for v in VRSTE if not TAKS.get(v)]}, ensure_ascii=False, indent=1))
ZIP = OUT / f'GljiveModel_v{VERZIJA}.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(PAKET.iterdir()): z.write(f, f.name)
print('✅ Gotovo:', ZIP, round(ZIP.stat().st_size / 1e6, 1), 'MB')
print('Pošalji Claudeu datoteku GljiveModel_v%d.zip iz mape GljiveModel na Google Driveu.' % VERZIJA)
try:
    from google.colab import files; files.download(str(ZIP))
except Exception:
    pass